# 04 — Préparer l’encodeur : audit du tokenizer CAMeLBERT

Nous préparons l’encodeur prévu au cahier des charges. Le **tokenizer** transforme chaque texte en unités puis en identifiants que le modèle recevra. Cet audit ne charge aucun poids et n’entraîne aucun classifieur. Contributions partielles à **L03, L05 et L12**.

CAMeLBERT utilise **WordPiece** : un mot peut être représenté par plusieurs morceaux. `##` indique une continuation, pas des caractères présents dans le texte d’origine. `[UNK]` signifie que le tokenizer n’a pas pu représenter une unité ; cela ne mesure pas la compréhension du modèle.

Prérequis : fichiers publics du tokenizer à la révision fixée déjà présents en local (`fetch_camelbert_tokenizer.py --download`), profil `sh scripts/tokenizer_uv.sh sync --locked`, corpus locaux autorisés et manifestes v2. Les cellules travaillent hors ligne ; elles ne téléchargent rien.

Garder le noyau **TuniScope (notebooks)**. Le code du tokenizer s’exécute dans son propre profil, sans changer les environnements classique ou BiGRU. Le mode par défaut **audit** mesure de nouveau toutes les lignes train/validation. **replay** vérifie un audit existant ; **off** n’exécute aucun outil.


In [1]:
from pathlib import Path
import os
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'scripts/tokenizer_uv.sh').is_file()), None)
if ROOT is None:
    raise RuntimeError('Ouvrir le notebook depuis le dépôt ou son dossier notebooks/.')
sys.path.insert(0, str(ROOT / 'scripts'))
from tokenizer_notebook_support import notebook_audit
from encoder_tokenizer_audit import readable_summary

AUDIT_MODE = os.environ.get('TUNISCOPE_TOKENIZER_MODE', 'audit')
REPLAY_AUDIT = ROOT / 'outputs/tokenizer_audits/2026-11-04_camelbert_raw_v2'
if AUDIT_MODE not in {'off', 'audit', 'replay'}:
    raise ValueError('Choisir off, audit ou replay.')
print('Mode effectif :', AUDIT_MODE.upper())


Mode effectif : AUDIT


## 1. Provenance et règles de mesure

Le modèle est `CAMeL-Lab/bert-base-arabic-camelbert-mix`, à la révision complète `9be352797bdf28a9ae21e2ae582aaaca7abdb22d`. Nous avons seulement sa fiche, ses configurations et le vocabulaire. La licence déclarée sur sa fiche est Apache-2.0.

Nous mesurons toutes les lignes des partitions v2 **train et validation**, sans sélectionner le test officiel. Le tokenizer conserve son vocabulaire préentraîné de 30 000 unités : nous ne le reconstruisons pas à partir de nos commentaires.

Les catégories AR/LATIN/MIXED/UNKNOWN décrivent l’écriture Unicode, pas la langue identifiée. Les groupes comptant moins de 100 exemples restent visibles, sans être artificiellement dupliqués.

## 2. Ce que nous mesurons

- Nombre de sous-mots par unité séparée par espaces : un proxy explicite, pas une annotation linguistique de mots.
- Longueurs, avec les marqueurs `[CLS]` et `[SEP]`, et percentiles P95/P99.
- Taux de tokens inconnus et proportion de textes concernés.
- Textes qui seraient tronqués à 64, 128, 256 ou 512 unités.

La règle choisit la plus petite longueur candidate couvrant au moins **99 % de chaque groupe d’écriture du train comptant au moins 100 lignes**. La validation ne choisit pas cette longueur.


In [2]:
audit_directory = None
audit_result = None
if AUDIT_MODE == 'off':
    print('Audit désactivé ; aucun corpus, tokenizer ou modèle chargé.')
else:
    audit_directory, audit_result = notebook_audit(AUDIT_MODE, REPLAY_AUDIT)
    print()
    print(readable_summary(audit_result))
    print('Dossier du rapport :', audit_directory)


Audit réel de tout le train/validation v2 ; aucun entraînement.

CAMeLBERT Mix — audit du tokenizer, sans poids ni entraînement
Révision : 9be352797bdf28a9ae21e2ae582aaaca7abdb22d
Texte brut ; train/validation v2, test officiel non sélectionné.
Corpus  Split       Écriture        N   Sous-mots/champ   UNK tokens    P99    Max
TSAC    train       AR           4426              1.58       0.49%     77    266
TSAC    train       LATIN        5355              3.88       0.06%     88    388
TSAC    train       MIXED         259              1.87       1.93%    113    189
TSAC    train       UNKNOWN        13              3.23       5.63%     32     32
TSAC    validation  AR            785              1.57       0.69%     94    176
TSAC    validation  LATIN         932              3.89       0.06%     81    140
TSAC    validation  MIXED          46              2.05       2.12%     54     54
TSAC    validation  UNKNOWN        11              4.18       2.17%     14     14
MADAR   train   

## 3. Lire les résultats

**Sous-mots/champ** : un rapport plus élevé signifie davantage de découpage par rapport aux unités séparées par espaces. **P99** : au moins 99 % des textes du groupe ont une longueur inférieure ou égale à cette valeur. **UNK tokens** : une proportion de tokens, pas un pourcentage de mots compris.

Une faible troncature signifie que presque tout le texte entre dans la fenêtre choisie. Elle ne garantit pas un bon sentiment ou une bonne identification dialectale. De même, représenter un emoji ou découper une négation ne prouve pas que l’encodeur saura les interpréter.

## 4. Examiner seulement des exemples inventés

Les textes suivants ne viennent pas des corpus. Ils servent à voir le découpage de la négation, de l’Arabizi, du français, des emojis et des marques arabes, sans publier de commentaire protégé.


In [3]:
if audit_result is not None:
    for probe in audit_result['invented_probes']:
        print(probe['name'], ':', repr(probe['text']))
        print('  Tokens :', probe['tokens'])
        if probe['normalizer_output'] != probe['text']:
            print('  Normaliseur interne :', repr(probe['normalizer_output']))
    counts = audit_result['parameter_counts']
    print()
    print('Paramètres théoriques de l’encodeur avec pooler :', counts['encoder_with_pooler'])
    print('Comptage depuis la configuration ; aucun checkpoint chargé.')
    print('Longueur proposée :', audit_result['length_decision']['max_length'])
    print('Rapport :', audit_directory)


arabizi : 'el service behi barcha'
  Tokens : ['e', '##l', 's', '##erv', '##ice', 'be', '##h', '##i', 'b', '##ar', '##ch', '##a']
negation_latine : 'el service moch behi'
  Tokens : ['e', '##l', 's', '##erv', '##ice', 'm', '##oc', '##h', 'be', '##h', '##i']
negation_tunisienne : 'الخدمة موش باهية'
  Tokens : ['الخدمة', 'مو', '##ش', 'باه', '##ية']
negation_attachee : 'ما عجبنيش'
  Tokens : ['ما', 'عجب', '##نيش']
francais : 'Le service est très agréable.'
  Tokens : ['Le', 's', '##erv', '##ice', 'e', '##st', 'tr', '##è', '##s', 'ag', '##r', '##é', '##able', '.']
melange : 'service behi ama livraison très lente'
  Tokens : ['s', '##erv', '##ice', 'be', '##h', '##i', 'am', '##a', 'l', '##iv', '##ra', '##is', '##on', 'tr', '##è', '##s', 'l', '##ent', '##e']
emojis : '🙂😍'
  Tokens : ['🙂', '##😍']
chiffres_arabizi : '7لو 3lech'
  Tokens : ['7', '##لو', '3', '##le', '##ch']
msa : 'أين المحطة؟'
  Tokens : ['أين', 'المحطة', '؟']
voyelles : 'بَاهي'
  Tokens : ['[UNK]']
repetition : 'باااهي برشا'
 

## 5. Rapport avec la suite

Avec la BiGRU, nous avions appris nous-mêmes une table de mots et ses vecteurs. Pour CAMeLBERT, le tokenizer et les futurs poids doivent correspondre à la même révision préentraînée. Cet audit prépare les entrées de ce futur modèle ; il ne fournit pas encore de score de classification.

La suite sera un contrôle de chargement et de faisabilité, puis l’encodeur **gelé** : ses poids seront conservés et seule une tête de classification sera apprise. Nous pourrons ensuite comparer avec le **fine-tuning**, qui adapte aussi les poids de l’encodeur. Le LLM/LoRA reste une autre expérience.

Le nombre de paramètres affiché est calculé depuis l’architecture BERT déclarée. Le contrôle d’un modèle réellement chargé reste à faire. Le test officiel, le budget GPU et les entraînements de L05 ne sont pas réalisés par ce notebook.

Documentation : [protocole](../docs/42_protocole_audit_tokenizer_encodeur.md). Les résultats de chaque relance sont conservés dans un nouveau dossier privé ; les résultats initiaux sont documentés séparément après vérification.
